In this notebook we detect trees and extract their heights.

Inputs: NDVI, DTM and DEM rasters

Processing steps (for every DSM file):
- calculate height of objects: DEM-DTM
- make a tree mask considering NDVI > 0.2 and height >2m
- mask the height raster: get a new height raster with only trees height
- reduce raster noise: apply max, median and mean filters
- simplify raster: make "blocks" every 1.5m
- extract the polygons for each different height of the raster
- saves a gdf with the results: 50 gdf cover all the city

Cleaning steps (for every gdf saved):
- remove polygons with height 0
- remove polygons with area <= 1.5m
- assigns CRS 4326 (neccesary for using pybdshadow library)
- explode multypoligons if there are
- simplify and smooth geometries

note: during all the steps, the temporary and previous files are removed to avoid memory issues (both disk and ram). Also every 20 iterations there is a sleep(30) to avoid memory overload

In [ ]:
!pip install geopandas shapely rasterio

In [ ]:
import os
import io
import json
import math
import time
import glob

import numpy as np
import pandas as pd
from tqdm import tqdm
import boto3

import geopandas as gpd
from shapely.geometry import box, shape, Polygon, MultiPolygon

import rasterio
from rasterio.crs import CRS
from rasterio.mask import mask
from rasterio.features import shapes
from rasterio.warp import reproject, Resampling
from rasterio.windows import from_bounds

from scipy.ndimage import maximum_filter, median_filter, uniform_filter

In [ ]:
# user-defined paths
base_dir = '../../../../datos-notebooks/3.global-data-cba/'
trees_folder_name = 'gch_cba'
city_boundary = 'cba_boundary.gpkg'
crs_epsg = 32720  # Projected CRS EPSG code, depending on the city

In [ ]:
# inputs
rasters_folder_path = os.path.join(base_dir, 'objects-height', trees_folder_name)
clipped_rasters_folder_path = os.path.join(base_dir, 'MRT-prediction/shadows/clipped-rasters/') 
trees_vectors_path = os.path.join(base_dir, 'MRT-prediction/shadows/trees-vectors/')
clean_trees_vectors_path = os.path.join(base_dir, 'MRT-prediction/shadows/trees-vectors-clean/')
bound_path = os.path.join(base_dir, 'vectors/',city_boundary)

In [ ]:
# Crop to 100m buffer from city boundary

os.makedirs(clipped_rasters_folder_path, exist_ok=True)

a = os.path.join(rasters_folder_path,os.listdir(rasters_folder_path)[0])
with rasterio.open(a) as src:
    crs = src.crs
    
bounds = gpd.read_file(bound_path)
bounds = bounds.to_crs(crs) # Ensure same crs 
bounds_buffer = bounds.buffer(100) # Buffer 100 meters


for file in os.listdir(rasters_folder_path):

    raster_path = os.path.join(rasters_folder_path, file)

    with rasterio.open(raster_path) as src:

        raster_bounds = box(*src.bounds) # Get raster bounding box as a polygon
        raster_gdf = gpd.GeoDataFrame(geometry=[raster_bounds], crs=src.crs)

        # CHECK IF RASTER INTERSECTS THE BUFFER AREA
        if not raster_gdf.intersects(bounds_buffer.geometry.union_all()).iloc[0]:

            print(f"Skipping {file} (no intersection).")
            continue

        print(f"Clipping {file} ...")

        # CLIP THE RASTER -- outside buffer area becomes 0
        tree_shapes = [bounds_buffer.geometry.union_all().__geo_interface__] # build mask geometry (single polygon)

        # clip raster (outside → 0)
        out_img, out_transform = mask(
            src,
            tree_shapes,
            crop=True,
            nodata=0,       # outside area becomes 0
            filled=True     # fill outside with nodata
        )

        # update metadata
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_img.shape[1],
            "width": out_img.shape[2],
            "transform": out_transform,
            "nodata": 0     # keep 0 as nodata 
        })

        # SAVE CLIPPED RASTER
        out_path = os.path.join(clipped_rasters_folder_path, f"clipped_{file}")

        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(out_img)

        print(f"Saved → {out_path}")


In [ ]:
# CONFIGURATION
height_list = sorted(glob.glob(os.path.join(clipped_rasters_folder_path, "*.tif")))
os.makedirs(trees_vectors_path, exist_ok=True)

In [ ]:
# loop through all clipped rasters to extract tree polygons --21minutes

for raster_path in height_list:

    filename = os.path.basename(raster_path)

    print(f"Processing {filename} ...")

    with rasterio.open(raster_path) as src:
        height = src.read(1)          
        transform = src.transform
        crs = src.crs

    # FILTER HEIGHTS > 2
    height = np.where(height > 2, height, 0)
    height_int = height.astype(np.int16)

    # CREATE 5 PART CONTAINERS
    results_parts = [[] for _ in range(5)]

    # VECTORIZE RASTER INTO POLYGONS
    for idx, (geom, val) in enumerate(shapes(height_int, transform=transform)):

        if val > 0:
            part_id = idx % 5  # distribute evenly
            results_parts[part_id].append({
                "height": float(val),
                "geometry": shape(geom),
                "filename": filename
            })

    # BUILD AND SAVE EACH PART
    for i, part_list in enumerate(results_parts):

        if not part_list:
            print(f"{filename} part{i+1}: EMPTY → skipped")
            continue

        gdf_part = gpd.GeoDataFrame(part_list, geometry="geometry", crs=crs)

        out_file = os.path.join(
            trees_vectors_path,
            f"{filename}_part{i+1}.geojson"
        )

        gdf_part.to_file(out_file, driver="GeoJSON")
        print(f"Saved {out_file}")


In [ ]:
# functions for smoothing results

def chaikin_smoothing(coords, refinements=2):
    for _ in range(refinements):
        new_coords = []
        for i in range(len(coords) - 1):
            p0 = np.array(coords[i])
            p1 = np.array(coords[i + 1])
            Q = 0.75 * p0 + 0.25 * p1
            R = 0.25 * p0 + 0.75 * p1
            new_coords.extend([Q, R])
        new_coords.append(new_coords[0])  # cerrar el anillo
        coords = new_coords
    return coords

def smooth_polygon(poly, refinements=2):
    if isinstance(poly, Polygon):
        smoothed = Polygon(chaikin_smoothing(list(poly.exterior.coords), refinements))
        return smoothed
    elif isinstance(poly, MultiPolygon):
        return MultiPolygon([smooth_polygon(p, refinements) for p in poly.geoms])
    else:
        return poly


In [ ]:
# Cleaning process for all split GeoJSONs (smoothing, simplification, filtering) -- 1.5h aprox
os.makedirs(clean_trees_vectors_path, exist_ok=True)

# Loop through every JSON file in the folder
for file in tqdm(os.listdir(trees_vectors_path)):

    part_file = os.path.join(trees_vectors_path, file)

    gdf = gpd.read_file(part_file)
    gdf = gdf.set_crs(crs_epsg, allow_override=True)    # overwrite CRS
    gdf = gdf[gdf['height'] != 0]                    # remove height=0
    gdf = gdf.explode(index_parts=False)             # explode geometries
    time.sleep(5)                                    # cooldown
    
    # smooth and simplify
    gdf['geometry'] = gdf['geometry'].apply(lambda geom: smooth_polygon(geom, refinements=5))
    gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.15, preserve_topology=True)

    # filter invalid or tiny geometries
    gdf = gdf[gdf.geometry.notnull()]
    gdf = gdf[gdf.geometry.area >= 1.5]
    gdf['geometry'] = gdf['geometry'].buffer(0)      # fix topology
    gdf = gdf.to_crs("EPSG:4326") # final CRS, global

    # save
    clean_name = file.replace(".json", "_clean.geojson")
    clean_path = os.path.join(clean_trees_vectors_path, clean_name)
    gdf.to_file(clean_path, driver="GeoJSON")

    # Optional: remove original partial file to free space
    # os.remove(part_file)
